# LinkedIn post classifier

Train a binary classifier: **useful = 1**, **useless = 0**.

In [9]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences


test_usefull = "../data/test/usefull_posts.cvs"
test_useless = "../data/test/useless_posts.cvs"
training_usefull = "../data/training/usefull_posts.cvs"
training_useless = "../data/training/useless_posts.cvs"

VectorSize = 300


## 1. Load and clean

Each post is one example, regardless of word count. Lowercasing and removing punctuation make the vocabulary simpler. The trade-off is that names like `C++` lose their symbols.


In [10]:
useful_train = pd.read_csv(training_usefull)['post_content'].dropna().astype(str).tolist()
useless_train = pd.read_csv(training_useless)['post_content'].dropna().astype(str).tolist()
useful_test = pd.read_csv(test_usefull)['post_content'].dropna().astype(str).tolist()
useless_test = pd.read_csv(test_useless)['post_content'].dropna().astype(str).tolist()

def clean(text):
    return ' '.join(re.findall(r'\b\w+\b', text.lower()))

texts = [clean(text) for text in useful_test + useful_train + useless_train + useless_test]
labels = np.array([1] * (len(useful_test) + len(useful_train)) + [0] * (len(useless_test) + len(useless_train)))
print(f'{len(useful_test) + len(useful_train)} useful, {len(useless_test) + len(useless_train)} useless posts')


489 useful, 440 useless posts


## 2. Split and encode

Reserve 20% for validation. `stratify` keeps the class proportions similar. Fit the tokenizer on training text only so validation vocabulary does not leak into training. Padding gives every post the same input length; longer posts are cut off.


In [11]:
train_texts, val_texts, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)
MAX_LEN = 350  # Room for most approximately 300-word posts.
tokenizer = Tokenizer(oov_token='<OOV>', filters='')
tokenizer.fit_on_texts(train_texts)

def encode(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

x_train, x_val = encode(train_texts), encode(val_texts)


## 3. Train Word2Vec

Word2Vec learns a 100-number vector for each word from its neighbors in the training posts. `min_count=1` keeps rare technical words, but also typos. The embedding matrix aligns vectors with tokenizer IDs; row 0 is padding.


In [12]:
w2v = Word2Vec([text.split() for text in train_texts], vector_size=VectorSize, min_count=0, epochs=10)
embeddings = np.zeros((len(tokenizer.word_index) + 1, 300), dtype=np.float32)
for word, index in tokenizer.word_index.items():
    if word in w2v.wv:
        embeddings[index] = w2v.wv[word]


## 4. Train the CNN

`Conv1D` looks for patterns across five neighboring words. Pooling retains strong pattern signals. The sigmoid output is the estimated probability that a post is useful. Training the embeddings lets them adapt to this task, but may overfit; early stopping restores the weights from the best validation-loss epoch.


In [13]:
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(len(embeddings), VectorSize, weights=[embeddings], trainable=True),
    Conv1D(64, 20, activation='relu'),
    MaxPooling1D(2),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(
    x_train, y_train, validation_data=(x_val, y_val),
    epochs=20, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
)


Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 128ms/step - accuracy: 0.5451 - loss: 0.8237 - val_accuracy: 0.6667 - val_loss: 0.6354
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 130ms/step - accuracy: 0.6931 - loss: 0.5911 - val_accuracy: 0.6559 - val_loss: 0.6520
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 125ms/step - accuracy: 0.7335 - loss: 0.5220 - val_accuracy: 0.7043 - val_loss: 0.5796
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 136ms/step - accuracy: 0.8452 - loss: 0.4203 - val_accuracy: 0.6828 - val_loss: 0.5552
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 122ms/step - accuracy: 0.8964 - loss: 0.3405 - val_accuracy: 0.6989 - val_loss: 0.5537
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 122ms/step - accuracy: 0.9462 - loss: 0.2354 - val_accuracy: 0.6505 - val_loss: 0.6277
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - accuracy: 0.9758 - loss: 0.1573 - val_accuracy: 0.7151 - val_loss: 0.5723


## 5. Evaluate and use

Accuracy is the overall fraction correct. Precision and recall show how the model performs for each class. A probability of 0.5 or higher is classified as useful.


In [14]:
useful_test = pd.read_csv(test_usefull)["post_content"].dropna().astype(str).tolist()
useless_test = pd.read_csv(test_useless)["post_content"].dropna().astype(str).tolist()

test_texts = [clean(text) for text in useful_test + useless_test]
x_test = encode(test_texts)  # Uses the tokenizer fitted on training data.
y_test = np.array([1] * len(useful_test) + [0] * len(useless_test))

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
predictions = (model.predict(x_test, verbose=0).ravel() >= 0.5).astype(int)

print(f"Test accuracy: {accuracy:.1%}")
print(classification_report(y_test, predictions, target_names=["useless", "useful"]))


Test accuracy: 91.0%
              precision    recall  f1-score   support

     useless       0.90      0.90      0.90       190
      useful       0.91      0.92      0.92       219

    accuracy                           0.91       409
   macro avg       0.91      0.91      0.91       409
weighted avg       0.91      0.91      0.91       409



In [15]:
model.save("../model_exports/linkedin_posts_classifier.keras")

In [16]:
from pathlib import Path

Path("../model_exports/tokenizer.json").write_text(tokenizer.to_json())

586875